# 365 Probabilidades - Dia #144
## Qual a probabilidade de a falta de tempo ser o que te afasta dos livros?

**Tipo:** Comportamento · Trilha do tempo, dia 5 de 24
**Data de publicação:** 2026-11-04
**Ferramenta:** Python
**Decisão analisada:** Se eu tivesse mais tempo, eu leria mais?
**Hashtag:** #365Probabilidades #Dia144 #ProbabilidadesDoTempo

---

### 📖 A História

Quase todo mundo tem um livro parado na mesa de cabeceira. Começado, com o marcador
no mesmo lugar há semanas. E quase todo mundo tem a mesma explicação pronta para ele.
Falta tempo. E por favor,
se você tiver outro motivo, por favor, me conte nos comentários para eu refazer a minha
percepção.

É a explicação mais dita no Brasil. Na pesquisa Retratos da Leitura, quem leu pouco
diz que foi por falta de tempo antes de qualquer outro motivo. Só que o que a gente diz
sobre o próprio dia e o que a gente faz com ele nem sempre combinam. Eu realmente acho
que gostaria de ter mais tempo para ler… mas, aplicando um grau maior de honestidade,
penso que me falta priorização.

Este dia da trilha do tempo testa a desculpa. Se a falta de tempo fosse o motivo, quem
tem o dia mais livre deveria ler muito mais do que quem tem o dia cheio. Fui ver nos
diários de um dia inteiro de mais de seis mil pessoas se isso acontece.

---

### 📚 O Conceito: o que se diz e o que se mede

Há duas formas de saber por que alguém não lê. A primeira é perguntar, e a resposta
vem com o que a pessoa acredita sobre si mesma. A segunda é olhar o dia dela, minuto a
minuto, e ver se a leitura aparece quando o tempo aparece.

A pesquisa americana de uso do tempo (American Time Use Survey, do governo dos Estados
Unidos) faz a segunda coisa. Cada pessoa conta o que fez nas 24 horas do dia anterior,
atividade por atividade. Não pergunta se ela lê. Registra se ela leu.

Com isso dá para separar duas perguntas que costumam vir juntas. Quantas pessoas
**dizem** que não leem por falta de tempo. E quanto da falta de leitura **some** quando
o dia fica livre.

---

### 🧮 O Modelo

Quatro camadas. As duas primeiras vêm das fontes. As duas últimas são contas nossas
sobre os microdados.

1. **O que se diz.** Retratos da Leitura no Brasil 2024: a proporção que culpa a falta
   de tempo, entre leitores e não leitores, com o fator ×0,80.
2. **A queda.** Bone et al. 2025: a proporção de americanos que leem por prazer num dia
   qualquer, de 2004 a 2023. Refazemos a conta com o ATUS 2025.
3. **O dia livre.** ATUS 2025, 6.146 diários: a chance de ler por prazer conforme as
   horas de obrigação do dia (trabalho, estudo, casa, cuidado e os deslocamentos ligados
   a eles), e para onde vai o tempo de quem tem o dia livre.
4. **A conta.** Regressão logística ponderada, com idade, sexo, escolaridade e fim de
   semana, e o contrafactual: quanto a leitura subiria se ninguém tivesse obrigação
   nenhuma naquele dia. Intervalo por reamostragem (bootstrap, 1.000 rodadas).

**Fonte central:**
- Bone, J. K., Bu, F., Sonke, J. K. & Fancourt, D. (2025). "The decline in reading for
  pleasure over 20 years of the American Time Use Survey". *iScience* 28(9), 113288,
  doi 10.1016/j.isci.2025.113288. ATUS 2003-2023 (sem 2020), **N = 236.270**. Leitura
  por prazer num dia médio caiu de **28% (2004)** para **16% (2023)**; razão de
  prevalência de 0,97 ao ano, IC 95% [0,97; 0,98].

**Segunda fonte (Brasil):**
- Instituto Pró-Livro, Itaú Cultural e Ipec (2024). *Retratos da Leitura no Brasil*,
  6ª edição. **5.504 entrevistas** em 208 municípios. Leitores 47%, não leitores 53%
  (primeira vez em que os não leitores são maioria). Entre os leitores, **46%** dizem
  que não leram mais por falta de tempo, e 75% gostariam de ter lido mais. Entre os não
  leitores, a falta de tempo é o primeiro motivo, com **33%**. Números lidos em
  divulgações da pesquisa, não no relatório (pendente).

**Dado próprio:**
- U.S. Bureau of Labor Statistics (2026). *American Time Use Survey 2025*, arquivo
  Activity Summary (atussum_2025.dat), **6.146 pessoas** de 15 anos ou mais, peso
  TUFINLWGT. Leitura por prazer = código 120312 ("Reading for personal interest").
  Conferência com o release oficial (25/06/2026): lazer e esporte 5,16 h/dia e TV
  2,61 h/dia.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEMENTE = 42
rng = np.random.default_rng(SEMENTE)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
RODAPE_ATUS = 'Fonte: U.S. Bureau of Labor Statistics, ATUS 2025 (6.146 diários) · cálculo próprio | #365Probabilidades'

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def jeffreys(k, n):
    return stats.beta(k + 0.5, n - k + 0.5)

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS DA LITERATURA E DO ARQUIVO ---
FATOR = 0.80   # só nas proporções autorrelatadas do Retratos; o diário do ATUS fica sem fator

# Retratos da Leitura no Brasil 2024 (6ª edição)
N_RETRATOS = 5504
P_LEITORES = 0.47
RETRATOS = {
    'leitores que culpam a falta de tempo':     0.46,
    'leitores que gostariam de ter lido mais':  0.75,
    'não leitores que culpam a falta de tempo': 0.33,
}
N_LEITORES_APROX = round(N_RETRATOS * P_LEITORES)          # aproximação declarada
N_NAO_LEITORES_APROX = N_RETRATOS - N_LEITORES_APROX

# Bone et al. 2025, iScience
BONE = {'n': 236270, 'p_2004': 0.28, 'p_2023': 0.16, 'rp_ano': 0.97, 'rp_ic': (0.97, 0.98)}

# ATUS 2025, arquivo Activity Summary (salvo nesta pasta)
atus = pd.read_csv('atussum_2025.dat')
w = atus['TUFINLWGT'].to_numpy(dtype=float)

def soma(prefixos):
    cols = [c for c in atus.columns if any(c.startswith(p) for p in prefixos)]
    return atus[cols].sum(axis=1).to_numpy() / 60

lazer   = soma(['t12', 't13', 't1812', 't1813'])                 # definição do BLS: lazer, esporte e deslocamento ligado a eles
obrig   = soma(['t02', 't03', 't04', 't05', 't06',
                't1802', 't1803', 't1804', 't1805', 't1806'])    # casa, cuidado, trabalho, estudo e deslocamentos ligados
ler_min = atus['t120312'].to_numpy(dtype=float)
leu     = (ler_min > 0).astype(float)
tv      = atus['t120303'].to_numpy() / 60
tela    = (atus['t120303'] + atus['t120304'] + atus['t120307'] + atus['t120308']).to_numpy() / 60
idade   = atus['TEAGE'].to_numpy(dtype=float)
mulher  = (atus['TESEX'] == 2).to_numpy(dtype=float)
superior = (atus['PEEDUCA'] >= 43).to_numpy(dtype=float)        # bacharelado ou mais
fds     = atus['TUDIARYDAY'].isin([1, 7]).to_numpy(dtype=float)

def media(x, m=None):
    return np.average(x, weights=w) if m is None else np.average(x[m], weights=w[m])

n_eff = w.sum() ** 2 / (w ** 2).sum()   # tamanho efetivo de Kish

print("=" * 70)
print("  DADOS")
print("=" * 70)
print(f"\n  Retratos da Leitura 2024 ({mil(N_RETRATOS)} entrevistas)")
for k, v in RETRATOS.items():
    print(f"    {k:<44} {vir(v * 100, 0)}%   ×0,80 → {vir(v * FATOR * 100)}%")
print(f"\n  Bone et al. 2025 (N = {mil(BONE['n'])}): {vir(BONE['p_2004'] * 100, 0)}% em 2004 → {vir(BONE['p_2023'] * 100, 0)}% em 2023 · RP {vir(BONE['rp_ano'], 2)} ao ano")
print(f"\n  ATUS 2025: {mil(len(atus))} diários · tamanho efetivo {mil(n_eff)}")
print(f"    Conferência com o BLS · lazer e esporte {vir(media(lazer), 2)} h/dia (release 5,16) · TV {vir(media(tv), 2)} h/dia (release 2,61)")
print(f"    Leram por prazer no dia:   {vir(media(leu) * 100)}%")
print(f"    Minutos de leitura, média geral {vir(media(ler_min), 0)} · entre quem leu {vir(media(ler_min, leu == 1), 0)}")
print(f"    Horas de obrigação, média  {vir(media(obrig))}")
print("=" * 70)

In [ ]:
# --- O MODELO ---

# CAMADA 1 · O que se diz (Retratos 2024), com Beta de Jeffreys sobre os N aproximados
print("=" * 70)
print("  CAMADA 1 · O QUE SE DIZ (Retratos da Leitura 2024)")
print("=" * 70)
camada1 = {}
for k, v in RETRATOS.items():
    n = N_LEITORES_APROX if k.startswith('leitores') else N_NAO_LEITORES_APROX
    ic = jeffreys(round(v * n), n).interval(0.95)
    camada1[k] = (v, ic)
    print(f"  {k:<44} {vir(v * 100, 0)}% · ×0,80 {vir(v * FATOR * 100)}% [{vir(ic[0] * FATOR * 100)}; {vir(ic[1] * FATOR * 100)}]  (n ≈ {mil(n)})")

# CAMADA 2 · A queda (Bone 2025) e a nossa conta de 2025
anos = np.arange(2004, 2026)
curva = BONE['p_2004'] * BONE['rp_ano'] ** (anos - 2004)
k_eff = media(leu) * n_eff
ic25 = jeffreys(k_eff, n_eff).interval(0.95)
print("\n" + "=" * 70)
print("  CAMADA 2 · A QUEDA")
print("=" * 70)
print(f"  Bone 2025: 28% em 2004 · 16% em 2023 · a curva de 0,97 ao ano dá {vir(curva[anos == 2023][0] * 100)}% em 2023")
print(f"  ATUS 2025 (nossa conta): {vir(media(leu) * 100)}% [{vir(ic25[0] * 100)}; {vir(ic25[1] * 100)}] · a curva previa {vir(curva[-1] * 100)}%")
print(f"  Num dia qualquer, cerca de 1 em cada 6 pessoas lê por prazer.")

# CAMADA 3 · O dia livre
faixas = [(-0.01, 1, 'até 1 h'), (1, 4, '1 a 4 h'), (4, 7, '4 a 7 h'), (7, 10, '7 a 10 h'), (10, 24, 'mais de 10 h')]
camada3 = []
print("\n" + "=" * 70)
print("  CAMADA 3 · O DIA LIVRE (horas de obrigação no dia do diário)")
print("=" * 70)
print(f"  {'obrigação':<14} {'pessoas':>8} {'% da pop.':>10} {'leram':>8} {'IC 95%':>16} {'tela (h)':>9} {'lazer (h)':>10}")
for a, b, rot in faixas:
    m = (obrig > a) & (obrig <= b)
    neff_m = w[m].sum() ** 2 / (w[m] ** 2).sum()
    p = media(leu, m); ic = jeffreys(p * neff_m, neff_m).interval(0.95)
    camada3.append((rot, m.sum(), w[m].sum() / w.sum(), p, ic, media(tela, m), media(lazer, m)))
    print(f"  {rot:<14} {m.sum():>8} {vir(w[m].sum() / w.sum() * 100):>9}% {vir(p * 100):>7}% [{vir(ic[0] * 100)}; {vir(ic[1] * 100)}] {vir(media(tela, m)):>8} {vir(media(lazer, m)):>9}")

livre, cheio = obrig <= 1, obrig > 7
dif_boot = []
idx = np.arange(len(atus))
for _ in range(2000):
    s = rng.choice(idx, len(idx))
    dif_boot.append(np.average(leu[s][livre[s]], weights=w[s][livre[s]]) - np.average(leu[s][cheio[s]], weights=w[s][cheio[s]]))
dif_ic = np.percentile(dif_boot, [2.5, 97.5])
print(f"\n  Dia livre (até 1 h) contra dia cheio (mais de 7 h): {vir(media(leu, livre) * 100)}% contra {vir(media(leu, cheio) * 100)}%")
print(f"  Diferença {vir((media(leu, livre) - media(leu, cheio)) * 100)} pontos, IC 95% [{vir(dif_ic[0] * 100)}; {vir(dif_ic[1] * 100)}]")
print(f"  Quem tem o dia livre passa {vir(media(tela, livre))} h em telas e {vir(media(ler_min, livre), 0)} min lendo, em média.")
nl = leu == 0
print(f"  Entre quem não leu: {vir(media(tela >= 0.5, nl) * 100, 0)}% passaram ao menos 30 min em telas; {vir(media(tela >= 2, nl) * 100, 0)}% ao menos 2 h.")

In [ ]:
# CAMADA 4 · A conta: regressão logística ponderada e o contrafactual do dia sem obrigação
def montar_X(obrig_):
    z = (idade - 45) / 10
    return np.column_stack([np.ones(len(z)), obrig_, z, z ** 2, mulher, superior, fds])
NOMES = ['constante', 'hora de obrigação', 'idade (por 10 anos)', 'idade²', 'mulher', 'superior completo', 'fim de semana']

def logit(X, y, pesos, iteracoes=60):
    wn = pesos / pesos.mean(); b = np.zeros(X.shape[1])
    for _ in range(iteracoes):
        p = 1 / (1 + np.exp(-X @ b))
        b = b + np.linalg.solve(X.T @ (X * (wn * p * (1 - p))[:, None]), X.T @ (wn * (y - p)))
    p = 1 / (1 + np.exp(-X @ b))
    H = np.linalg.inv(X.T @ (X * (wn * p * (1 - p))[:, None])); S = X * (wn * (y - p))[:, None]
    return b, np.sqrt(np.diag(H @ S.T @ S @ H))   # erro-padrão robusto (sanduíche)

def prever(b, obrig_, s=None):
    s = np.arange(len(atus)) if s is None else s
    X = montar_X(obrig_)[s]
    return np.average(1 / (1 + np.exp(-X @ b)), weights=w[s])

X = montar_X(obrig)
b, se = logit(X, leu, w)
print("=" * 70)
print("  CAMADA 4 · A CONTA (logística ponderada, erro-padrão robusto)")
print("=" * 70)
for n_, bb, ss in zip(NOMES, b, se):
    if n_ != 'constante':
        print(f"  {n_:<22} razão de chances {vir(np.exp(bb), 3)} [{vir(np.exp(bb - 1.96 * ss), 3)}; {vir(np.exp(bb + 1.96 * ss), 3)}]")

p_hoje = prever(b, obrig)
p_zero = prever(b, np.zeros_like(obrig))
p_dez  = prever(b, np.full_like(obrig, 10.0))
explicada = (p_zero - p_hoje) / (1 - p_hoje)

boot = []
for _ in range(1000):
    s = rng.choice(idx, len(idx))
    bs, _ = logit(X[s], leu[s], w[s], iteracoes=25)
    ph = prever(bs, obrig, s); pz = prever(bs, np.zeros_like(obrig), s)
    boot.append((ph, pz, (pz - ph) / (1 - ph)))
boot = np.array(boot)
ic_z = np.percentile(boot[:, 1], [2.5, 97.5]); ic_e = np.percentile(boot[:, 2], [2.5, 97.5])

print(f"\n  Leitura prevista com as obrigações de hoje:  {vir(p_hoje * 100)}%")
print(f"  Se ninguém tivesse obrigação nenhuma no dia: {vir(p_zero * 100)}% [{vir(ic_z[0] * 100)}; {vir(ic_z[1] * 100)}]")
print(f"  Se todo mundo tivesse 10 h de obrigação:     {vir(p_dez * 100)}%")
print(f"\n  Parcela de quem não lê que passaria a ler com o dia inteiro livre: {vir(explicada * 100)}% [{vir(ic_e[0] * 100)}; {vir(ic_e[1] * 100)}]")
print(f"  Ou {vir(explicada * 100, 0)} em cada 100 que não leem. Os outros {vir(100 - explicada * 100, 0)} continuariam sem ler.")
print(f"\n  Comparação: superior completo multiplica a chance de ler por {vir(np.exp(b[5]), 1)}; um dia inteiro livre (−{vir(media(obrig))} h) por {vir(np.exp(-b[1] * media(obrig)), 2)}.")

In [ ]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 · O que se diz
fig1, ax1 = plt.subplots(figsize=(12, 8))
rotulos = list(RETRATOS.keys()); y = np.arange(len(rotulos))
pub = np.array([RETRATOS[k] for k in rotulos]) * 100
ax1.barh(y, pub, color=DOURADO, alpha=0.3, height=0.55, label='publicado')
ax1.barh(y, pub * FATOR, color=DOURADO, alpha=0.95, height=0.55, label='com o fator ×0,80')
for i, k in enumerate(rotulos):
    lo, hi = camada1[k][1]
    ax1.plot([lo * FATOR * 100, hi * FATOR * 100], [i, i], color='#333', lw=2)
    ax1.text(pub[i] + 1, i, f"{vir(pub[i], 0)}%  →  {vir(pub[i] * FATOR)}%", va='center', fontsize=13, fontweight='bold', color='#6b5a1e')
ax1.set_yticks(y); ax1.set_yticklabels([r[0].upper() + r[1:] for r in rotulos], fontsize=12); ax1.invert_yaxis()
ax1.set_xlim(0, 100); ax1.set_xlabel('% das pessoas', fontsize=12)
ax1.set_title('A falta de tempo é a explicação mais dita no Brasil\nRetratos da Leitura no Brasil 2024, 5.504 entrevistas', fontsize=14, pad=16)
ax1.legend(frameon=False, fontsize=11, loc='lower right')
plt.figtext(0.5, 0.01, 'Fonte: Instituto Pró-Livro, Itaú Cultural e Ipec, Retratos da Leitura no Brasil 2024 · fator ×0,80 e IC aproximado | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-144-grafico-01-o-que-se-diz.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · A queda
fig2, ax2 = plt.subplots(figsize=(12, 8))
ax2.plot(anos, curva * 100, color=CINZA, lw=2, ls='--', label='queda de 3% ao ano (razão de prevalência 0,97)')
ax2.scatter([2004, 2023], [28, 16], s=160, color=VERMELHO, zorder=3, label='Bone et al. 2025')
ax2.errorbar([2025], [media(leu) * 100], yerr=[[media(leu) * 100 - ic25[0] * 100], [ic25[1] * 100 - media(leu) * 100]],
             fmt='o', ms=12, color=VERDE, capsize=6, lw=2, zorder=3, label='ATUS 2025, conta nossa (IC 95%)')
ax2.text(2004.3, 28.6, '28%', fontsize=15, fontweight='bold', color=VERMELHO)
ax2.text(2022.2, 17.3, '16%', fontsize=15, fontweight='bold', color=VERMELHO)
ax2.text(2024.2, media(leu) * 100 - 2.2, f"{vir(media(leu) * 100)}%", fontsize=13, fontweight='bold', color=VERDE)
ax2.set_ylim(0, 35); ax2.set_xlim(2003, 2026.5)
ax2.set_ylabel('Leram por prazer num dia qualquer (%)', fontsize=12)
ax2.set_title('Em vinte anos, a leitura por prazer perdeu quatro em cada dez leitores diários\nEstados Unidos, diários de um dia inteiro', fontsize=14, pad=16)
ax2.legend(frameon=False, fontsize=11, loc='upper right')
plt.figtext(0.5, 0.01, 'Fonte: Bone, Bu, Sonke & Fancourt (2025), iScience 28(9), N = 236.270 · BLS, ATUS 2025 · cálculo próprio | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-144-grafico-02-a-queda.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · O dia livre
fig3, ax3 = plt.subplots(figsize=(12, 8))
x = np.arange(len(camada3))
pl = np.array([c[3] for c in camada3]) * 100
lo = np.array([c[4][0] for c in camada3]) * 100; hi = np.array([c[4][1] for c in camada3]) * 100
ax3.bar(x, pl, color=VERDE, alpha=0.9, width=0.55)
ax3.errorbar(x, pl, yerr=[pl - lo, hi - pl], fmt='none', ecolor='#333', capsize=6, lw=1.5)
for i, c in enumerate(camada3):
    ax3.text(i, hi[i] + 0.8, f"{vir(pl[i])}%", ha='center', fontsize=14, fontweight='bold', color=VERDE)
    ax3.text(i, 1.2, f"{vir(c[5])} h\nde tela", ha='center', fontsize=11, color='white', fontweight='bold')
ax3.set_xticks(x); ax3.set_xticklabels([c[0] for c in camada3], fontsize=12)
ax3.set_xlabel('Horas de obrigação no dia (trabalho, estudo, casa, cuidado e deslocamentos)', fontsize=12)
ax3.set_ylabel('Leram por prazer naquele dia (%)', fontsize=12); ax3.set_ylim(0, 30)
ax3.set_title('Quem tem o dia livre não lê mais do que quem tem o dia cheio\no tempo que sobra vai para a tela', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE_ATUS, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-144-grafico-03-o-dia-livre.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · A conta (distribuição bootstrap da parcela explicada pelo tempo)
fig4, ax4 = plt.subplots(figsize=(12, 8))
dados = boot[:, 2] * 100
ax4.hist(dados, bins=40, color=DOURADO, alpha=0.85, edgecolor='white')
ax4.axvline(explicada * 100, color='#333', lw=2.5)
for v in ic_e * 100:
    ax4.axvline(v, color=VERMELHO, ls='--', lw=2)
ax4.text(explicada * 100 + 0.1, ax4.get_ylim()[1] * 0.9, f"{vir(explicada * 100)}%\nIC 95% [{vir(ic_e[0] * 100)}; {vir(ic_e[1] * 100)}]",
         fontsize=13, fontweight='bold')
ax4.set_xlabel('De cada 100 pessoas que não leram, quantas leriam com o dia inteiro livre', fontsize=12)
ax4.set_ylabel('Rodadas de reamostragem (de 1.000)', fontsize=12)
ax4.set_title('Tirar todas as obrigações do dia traz poucos leitores de volta\nlogística ponderada com idade, sexo, escolaridade e fim de semana', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE_ATUS, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-144-grafico-04-a-conta.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**A falta de tempo explica pouco. Se ninguém tivesse obrigação nenhuma no dia, a leitura
subiria de 16,1% para 19,1%. De cada 100 pessoas que não leram, umas 4 passariam a ler
(IC 95% de 1,4 a 6,0). As outras 96 continuariam sem ler, com o dia inteiro livre.**

No Brasil, 46% dos leitores dizem que leram menos por falta de tempo (36,8% com o fator
×0,80). É a explicação mais dita. Nos diários americanos, quem tinha até uma hora de
obrigação no dia leu tanto quanto quem tinha mais de sete, 13% contra 13%. A diferença
entre os dois grupos não está no tempo livre. Está no que se faz com ele. Quem tem o dia
livre passa 6 horas em telas e lê, em média, 22 minutos.

O tempo pesa um pouco, e a conta mostra isso. Cada hora de obrigação diminui as chances de
ler em 3,5%, depois de descontar idade, sexo e escolaridade. Mas ter curso superior multiplica
a chance de ler por três, e um dia inteiramente livre multiplica por 1,26. O hábito pesa
mais do que a agenda.

*Se amanhã o seu dia viesse vazio, o livro da mesa de cabeceira seria a primeira coisa
que você pegaria?*

---

### ⚠️ Limitações do Modelo

- O diário é de um dia só. Quem lê duas vezes por semana aparece como leitor em alguns
  dias e não leitor em outros. A conta fala de leitura num dia qualquer, não de hábito.
- Os dados de uso do tempo são dos Estados Unidos. O Brasil não tem pesquisa de uso do
  tempo com microdados abertos; a ponte com o Brasil é o Retratos da Leitura, que mede o
  que as pessoas dizem, não o que fazem no dia.
- Os números do Retratos foram lidos em divulgações da pesquisa, não no relatório. Os N
  de leitores e não leitores são aproximados (47% e 53% de 5.504), e os intervalos da
  camada 1 são só uma ordem de grandeza.
- O fator ×0,80 entra só nas proporções do Retratos (autorrelato). O diário do ATUS
  registra atividades e fica sem fator.
- Regressão não é causa. Quem tem o dia livre é diferente de quem tem o dia cheio em
  coisas que a regressão não vê (saúde, renda, rotina). O contrafactual do dia sem
  obrigação é uma conta, não um experimento.
- Não usamos os pesos replicados do BLS; o intervalo vem de reamostragem simples das
  pessoas com o peso final. A coleta parou de 1º/10 a 12/11/2025 por causa da
  paralisação do governo americano.
- "Ler por prazer" no ATUS inclui livro, revista e jornal, em papel, tela ou áudio, como
  atividade principal. Ler no celular entre uma coisa e outra pode não ser registrado.
- Corda com o #088 (papel contra tela), objeto diferente: lá se media a compreensão, aqui
  a ocorrência.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*